# OSM: barrios

Análisis exploratorio independiente de bronce. Fecha de revisión: 13 de noviembre de 2026.

Se examinan los archivos locales originales, no una tabla Delta ejecutada en Databricks.
No se limpia ni sobrescribe la fuente. Las transformaciones que aparezcan se hacen en copias
para evaluar lo que necesitará plata. Los resultados no prueban causalidad ni certifican seguridad.

Cada notebook sigue el orden: procedencia y unidad, calidad por columna y geometría,
distribuciones, límites y reglas para plata. No se muestran registros individuales de edificios oficiales.

In [ ]:
from pathlib import Path
import sys, json, re, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "CLAUDE.md").exists() and (p / "data/externos").is_dir())
sys.path.insert(0, str(ROOT / "analisis_bronce"))
from soporte import (ficha, huella, leer_geojson, mostrar_perfil, geometria, conteos,
                     faltantes, guardar_resumen, grafico_barras, SALIDAS, CRS_METROS)
pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 60)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})

## 1. Procedencia y unidad de análisis

Archivo: `data/externos/osm_barrios.geojson`. Una fila representa un objeto de límite territorial de OpenStreetMap.
La fecha de descarga no equivale a fecha de verificación del objeto en terreno.

In [ ]:
PATH = ROOT / 'data/externos/osm_barrios.geojson'
meta = ficha(PATH)
raw, df, g = leer_geojson(PATH)
print("Archivo:", meta["archivo"], "| filas:", len(df), "| propiedades:", len(df.columns))
display(pd.Series(meta, name="procedencia").to_frame())

## 2. Calidad por columna y geometría

In [ ]:
calidad = mostrar_perfil(df)
geo = geometria(g)
display(pd.Series(geo, name="geometria").to_frame())
resumen = {"dataset": '03_osm_barrios', **meta, "filas": len(df), "columnas": list(df.columns),
           "calidad_columnas": calidad.to_dict("records"), "geometria": geo}

## 3. Identidad, topología y cobertura de los límites

In [ ]:
identidad = pd.Series({"osm_id_nulos": int(df.osm_id.isna().sum()),
                       "osm_id_duplicados": int(df.osm_id.duplicated().sum()),
                       "nombres_nulos": int(df["name"].isna().sum()),
                       "nombres_repetidos": int(df["name"].duplicated().sum())})
display(identidad.to_frame("valor"))
metrica = g.to_crs(CRS_METROS)
areas = metrica.geometry.area
union = metrica.geometry.union_all()
area_stats = {"suma_area_km2": float(areas.sum()/1e6), "area_union_km2": float(union.area/1e6),
              "solape_interno_km2": float(max(0,areas.sum()-union.area)/1e6),
              "area_min_km2":float(areas.min()/1e6),"area_max_km2":float(areas.max()/1e6)}
display(pd.Series(area_stats).to_frame("valor"))
for c in ["admin_level","boundary","type"]:
 if c in df:display(conteos(df[c],c))
campos = faltantes(df,["osm_id","name","ref","comuna","source"])
display(campos)
fig,ax=plt.subplots(figsize=(8,6));g.plot(ax=ax,facecolor="#e4eef0",edgecolor="#226e86",linewidth=.7)
ax.set(title="Geometría original de límites OSM",xlabel="Longitud",ylabel="Latitud")
fig.tight_layout();fig.savefig(SALIDAS/(resumen["dataset"]+"_limites.png"),dpi=140);plt.show()
resumen.update({"identidad":identidad.to_dict(),"areas":area_stats,"disponibilidad":campos.to_dict("records")})

## Reglas decididas para plata

- **Incorporar como auxiliar condicionado a validar el cruce con comunas.** El campo `comuna` es opcional y no puede ser la única llave.
- Conservar nombre original y `osm_id`; normalizar espacios en una columna nueva. Un nombre repetido no acredita duplicación del barrio.
- Comparar cobertura y solapes con la unión de comunas en la síntesis. No borrar fuera de cobertura antes de contar y documentar la exclusión.
- El código catastral BARRIO del inventario no es automáticamente el `osm_id` ni el nombre OSM.

In [ ]:
assert huella(PATH) == meta["sha256"], "La fuente cruda cambió durante el análisis"
guardar_resumen('03_osm_barrios', resumen)
print("Resumen y gráficos guardados; la fuente cruda conserva su SHA-256.")